In [1]:
import optuna
from optuna.samplers import TPESampler
from optuna.trial import Trial
from optuna.study import create_study
import diff_cont
import lambda_cont
import libs.agent_infra as ai
import os
import json
import datetime
import itertools
import constants as Cs
import concurrent.futures
import numpy as np
from concurrent.futures import ProcessPoolExecutor

/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-15 22:02:44.253988: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-15 22:02:44.286043: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-15 22:02:44.998939: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF

In [ ]:


SEEDS = [101, 102, 103]
TEST_EVAL_EPS = 6

def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    cross_method = trial.suggest_categorical("crossmethod", ["uniform", "mean"])
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    m = trial.suggest_categorical("mu",[40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.1, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    cross_uni = None
    if cross_method == "uniform":
        cross_uni = trial.suggest_float("cross", 0.1, 0.9, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.lambda_cont.argumented_function, enviroment=environment,
                container="fitness",
                cross_method=cross_method,
                ng = 15,
                l = l,
                m = m,
                cr = cr,
                mr = mr,
                mutation_sigma = sigma,
                cross_uni = cross_uni if cross_uni is not None else 0.5,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler()
study = create_study(direction="maximize", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/lambda.db", load_if_exists=True)
study.optimize(objective, n_trials=100, n_jobs=5)
print(study.best_trials)

[I 2026-05-15 22:03:14,781] A new study created in RDB with name: no-name-e88fa580-a490-41e6-bbb1-5aa576e4e219


gen	nevals	avg     	min     	max     	std    
0  	40    	-440.424	-862.095	-125.803	180.751
gen	nevals	avg     	min     	max     	std    
0  	40    	-440.424	-862.095	-125.803	180.751
gen	nevals	avg     	min     	max    	std    
0  	40    	-485.637	-1396.75	-100.56	241.433
gen	nevals	avg     	min     	max    	std    
0  	40    	-485.637	-1396.75	-100.56	241.433
gen	nevals	avg     	min     	max     	std    
0  	40    	-481.335	-1444.58	-67.5599	265.169
gen	nevals	avg     	min     	max     	std    
0  	40    	-481.335	-1444.58	-67.5599	265.169
gen	nevals	avg     	min     	max     	std    
0  	50    	-420.759	-862.095	-125.803	182.866
gen	nevals	avg     	min     	max    	std    
0  	50    	-499.444	-1396.75	-100.56	244.059
gen	nevals	avg     	min     	max     	std    
0  	50    	-478.436	-1444.58	-67.5599	247.187
gen	nevals	avg     	min     	max     	std    
0  	60    	-439.942	-862.095	-125.803	174.319
gen	nevals	avg     	min     	max     	std    
0  	60    	-439.942	-862.095	-125.803	17

[I 2026-05-15 22:29:57,353] Trial 3 finished with value: -86.1212442655203 and parameters: {'crossmethod': 'mean', 'lambda': 40, 'mu': 50, 'mutation_rate': 0.03, 'cross_rate': 0.5}. Best is trial 3 with value: -86.1212442655203.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Fi

gen	nevals	avg     	min     	max     	std    
0  	50    	-420.759	-862.095	-125.803	182.866
gen	nevals	avg     	min     	max    	std    
0  	50    	-499.444	-1396.75	-100.56	244.059
gen	nevals	avg     	min     	max     	std    
0  	50    	-478.436	-1444.58	-67.5599	247.187
14 	47    	-13.4429	-13.4429	-13.4429	1.77636e-15
1  	37    	-233.006	-494.086	-112.391	122.365
1  	30    	-315.992	-610.143	-100.56	152.173
1  	33    	-299.418	-567.77 	-58.2528	139.929
2  	35    	-156.015	-394.958	-85.1474	66.5422
2  	35    	-231.605	-471.202	-95.0211	116.982
2  	33    	-189.457	-395.252	-58.2528	101.045
3  	34    	-139.32 	-460.454	-110.508	59.5015
3  	32    	-145.885	-442.677	-95.0211	75.5839
3  	33    	-144.255	-388.299	-58.2528	84.614 
4  	37    	-135.682	-396.708	-110.508	53.8763
4  	34    	-129.408	-472.915	-95.0211	68.9886
4  	39    	-113.363	-290.2  	-58.2528	61.6539
5  	38    	-105.819	-194.506	-84.7616	19.1479
5  	36    	-117.902	-125.803	-110.508	7.16312
5  	34    	-94.1603	-227.868	-58.

[I 2026-05-15 22:35:05,290] Trial 2 finished with value: -71.31268225508906 and parameters: {'crossmethod': 'mean', 'lambda': 60, 'mu': 40, 'mutation_rate': 0.09, 'cross_rate': 0.5}. Best is trial 2 with value: -71.31268225508906.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named '

...

9  	48    	-67.5814	-80.2121	-38.6856	10.0534
7  	52    	-104.276	-143.737	-37.0239	25.0483
15 	46    	-2.92591	-2.92591	-2.92591	4.44089e-16
12 	48    	-52.3315	-145.126	38.4365 	29.2268
9  	46    	-70.1584	-70.1584	-70.1584	1.42109e-14
10 	49    	-62.0481	-276.987	-30.6351	51.115 
13 	49    	-69.9521	-70.1584	-68.6847	0.511368   
10 	46    	-62.3054	-80.2121	-38.6856	11.451 
8  	53    	-101.799	-380.085	-37.0239	57.4511
10 	49    	-70.1584	-70.1584	-70.1584	1.42109e-14
14 	42    	-69.8047	-70.1584	-68.6847	0.629409   
13 	50    	-49.0455	-141.202	38.4365 	29.2824
11 	47    	-45.2897	-232.028	-30.6351	32.0425
11 	50    	-67.8531	-290.554	-38.6856	40.2585
11 	51    	-70.129 	-70.1584	-68.6847	0.206323   
9  	51    	-78.0602	-275.625	-30.6351	48.1476
15 	47    	-69.5689	-70.1584	-68.6847	0.721982   
12 	43    	-35.5626	-37.0239	-22.8505	3.40797
12 	48    	-52.3315	-145.126	38.4365 	29.2268
14 	51    	-38.7474	-58.74  	61.4876 	23.4484
12 	46    	-70.07  	-70.1584	-68.6847	0.349994   
10

[I 2026-05-16 06:21:55,124] Trial 97 finished with value: -77.47776315161146 and parameters: {'crossmethod': 'uniform', 'lambda': 50, 'mu': 60, 'mutation_rate': 0.03, 'cross_rate': 0.7}. Best is trial 55 with value: 154.1307686570909.


14 	47    	-32.622 	-37.0239	-22.8505	4.4251 
15 	47    	-34.6108	-215.836	-22.8505	26.091 


[I 2026-05-16 06:25:40,621] Trial 98 finished with value: 110.50841240833573 and parameters: {'crossmethod': 'uniform', 'lambda': 50, 'mu': 60, 'mutation_rate': 0.01, 'cross_rate': 0.8}. Best is trial 55 with value: 154.1307686570909.
[I 2026-05-16 06:26:22,896] Trial 99 finished with value: 110.50841240833573 and parameters: {'crossmethod': 'uniform', 'lambda': 50, 'mu': 60, 'mutation_rate': 0.01, 'cross_rate': 0.8}. Best is trial 55 with value: 154.1307686570909.


[FrozenTrial(number=55, state=<TrialState.COMPLETE: 1>, values=[154.1307686570909], datetime_start=datetime.datetime(2026, 5, 16, 3, 1, 29, 117665), datetime_complete=datetime.datetime(2026, 5, 16, 3, 41, 3, 321856), params={'crossmethod': 'uniform', 'lambda': 50, 'mu': 50, 'mutation_rate': 0.03, 'cross_rate': 0.8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'crossmethod': CategoricalDistribution(choices=('uniform', 'mean')), 'lambda': CategoricalDistribution(choices=(30, 40, 50, 60)), 'mu': CategoricalDistribution(choices=(30, 40, 50, 60)), 'mutation_rate': FloatDistribution(high=0.1, log=False, low=0.0, step=0.01), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1)}, trial_id=56, value=None)]


In [4]:
study.best_trials

[FrozenTrial(number=55, state=<TrialState.COMPLETE: 1>, values=[154.1307686570909], datetime_start=datetime.datetime(2026, 5, 16, 3, 1, 29, 117665), datetime_complete=datetime.datetime(2026, 5, 16, 3, 41, 3, 321856), params={'crossmethod': 'uniform', 'lambda': 50, 'mu': 50, 'mutation_rate': 0.03, 'cross_rate': 0.8}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'crossmethod': CategoricalDistribution(choices=('uniform', 'mean')), 'lambda': CategoricalDistribution(choices=(30, 40, 50, 60)), 'mu': CategoricalDistribution(choices=(30, 40, 50, 60)), 'mutation_rate': FloatDistribution(high=0.1, log=False, low=0.0, step=0.01), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1)}, trial_id=56, value=None)]